# 04 — Patient-level malignant pseudobulk condition reanalysis

**Purpose.** Reassess condition-associated expression after making the independent biological unit the source-backed patient rather than the regional sample. Raw sample pseudobulk counts from Notebook 22 are summed across regions belonging to the same patient before library-size normalization.

This is a sensitivity analysis. It does not overwrite canonical Notebook 22 outputs. The resulting condition comparisons remain exploratory because LGG has only two independent patients and because GBM versus the DS3 conditions is confounded with dataset of origin.

Run this notebook from top to bottom in a fresh kernel. All notebook text and outputs intended for the repository are in English.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

NOTEBOOK_NAME = "04_patient_level_malignant_pseudobulk_condition_reanalysis.ipynb"
CONDITION_ORDER = ["GBM", "LGG", "ndGBM", "rGBM"]
MIN_CPM = 1.0
MIN_PATIENTS_EXPRESSED = 3
MIN_PATIENTS_PER_TESTED_GROUP = 3

DESIGN_REL = Path("metadata/sensitivity/source_backed_study_design/source_backed_sample_design.tsv")
COUNTS_REL = Path("metadata/condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_counts_pseudobulk.tsv.gz")
QC_REL = Path("metadata/condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_sample_QC.tsv")
HISTORICAL_SUMMARY_REL = Path("metadata/condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_test_summary.tsv")
OUT_REL = Path("metadata/sensitivity/patient_level_malignant_pseudobulk_condition_reanalysis")

def resolve_root():
    candidates = []
    if os.environ.get("GLIOMA_PROJECT_ROOT"):
        candidates.append(Path(os.environ["GLIOMA_PROJECT_ROOT"]).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents, Path(r"G:\Repository\glioma-cnv-single-cell-atlas")])
    for candidate in candidates:
        if all((candidate / rel).exists() for rel in [DESIGN_REL, COUNTS_REL, QC_REL, HISTORICAL_SUMMARY_REL]):
            return candidate.resolve()
    raise FileNotFoundError("Could not resolve the repository root or required inputs.")

ROOT = resolve_root()
OUT_DIR = ROOT / OUT_REL
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Output directory:", OUT_DIR)


Repository root: G:\Repository\glioma-cnv-single-cell-atlas
Output directory: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\patient_level_malignant_pseudobulk_condition_reanalysis


In [2]:
def bh(values):
    values = np.asarray(values, dtype=float)
    result = np.full(values.shape, np.nan)
    valid = np.isfinite(values)
    if valid.any():
        result[valid] = multipletests(values[valid], method="fdr_bh")[1]
    return result

def safe_welch(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    if np.std(x) == 0 and np.std(y) == 0:
        return 1.0 if np.mean(x) == np.mean(y) else np.nan
    return float(stats.ttest_ind(x, y, equal_var=False).pvalue)

def safe_mannwhitney(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    return float(stats.mannwhitneyu(x, y, alternative="two-sided", method="auto").pvalue)

def rank_biserial(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    u = stats.mannwhitneyu(x, y, alternative="two-sided", method="auto").statistic
    return float(2 * u / (len(x) * len(y)) - 1)

def counts_to_logcpm(counts):
    array = counts.to_numpy(dtype=float)
    libraries = array.sum(axis=1)
    if np.any(libraries <= 0):
        raise AssertionError("A patient pseudobulk library has zero total counts.")
    return pd.DataFrame(np.log2(array / libraries[:, None] * 1e6 + 1.0), index=counts.index, columns=counts.columns)

def gene_filter(counts):
    array = counts.to_numpy(dtype=float)
    libraries = array.sum(axis=1)
    cpm = array / libraries[:, None] * 1e6
    return pd.Series((cpm >= MIN_CPM).sum(axis=0) >= MIN_PATIENTS_EXPRESSED, index=counts.columns)

def write_tsv(frame, name, index=False, compression=None):
    path = OUT_DIR / name
    frame.to_csv(path, sep="	", index=index, compression=compression)
    outputs[name] = path
    return path


In [3]:
design = pd.read_csv(ROOT / DESIGN_REL, sep="	")
sample_qc = pd.read_csv(ROOT / QC_REL, sep="	")
sample_counts = pd.read_csv(ROOT / COUNTS_REL, sep="	", index_col=0)

required_design = {"sample_id", "condition", "core_dataset", "source_backed_patient_id", "source_patient_label"}
missing = required_design - set(design.columns)
if missing:
    raise AssertionError(f"Source-backed design columns are missing: {sorted(missing)}")
if design["sample_id"].duplicated().any():
    raise AssertionError("Source-backed design contains duplicate sample IDs.")
if sample_counts.index.duplicated().any() or sample_qc["sample_id"].duplicated().any():
    raise AssertionError("Notebook 22 pseudobulk inputs contain duplicate sample IDs.")
if set(sample_counts.index.astype(str)) != set(sample_qc["sample_id"].astype(str)):
    raise AssertionError("Notebook 22 count and QC sample sets differ.")

meta = sample_qc.merge(
    design[["sample_id", "condition", "core_dataset", "source_backed_patient_id", "source_patient_label"]],
    on="sample_id", how="left", validate="one_to_one", suffixes=("_nb22", "_source")
)
if meta["source_backed_patient_id"].isna().any():
    raise AssertionError("One or more Notebook 22 samples do not map to the source-backed design.")
if not meta["condition_nb22"].astype(str).equals(meta["condition_source"].astype(str)):
    raise AssertionError("Condition labels differ between Notebook 22 and the source-backed design.")

meta = (meta.set_index("sample_id")
        .reindex(sample_counts.index.astype(str))
        .rename_axis("sample_id")
        .reset_index())
sample_to_patient = meta.set_index("sample_id")["source_backed_patient_id"]
patient_counts = sample_counts.groupby(sample_to_patient, sort=True).sum()

patient_meta = (meta.groupby("source_backed_patient_id", as_index=False)
    .agg(source_patient_label=("source_patient_label", "first"),
         condition=("condition_source", "first"),
         core_dataset=("core_dataset", "first"),
         n_regional_samples=("sample_id", "nunique"),
         n_cells=("n_cells", "sum")))
if (meta.groupby("source_backed_patient_id")["condition_source"].nunique() > 1).any():
    raise AssertionError("A source-backed patient spans more than one condition.")
if (meta.groupby("source_backed_patient_id")["core_dataset"].nunique() > 1).any():
    raise AssertionError("A source-backed patient spans more than one core dataset.")

patient_meta = patient_meta.set_index("source_backed_patient_id").loc[patient_counts.index].reset_index()
patient_meta["library_size"] = patient_counts.sum(axis=1).to_numpy(dtype=float)
patient_meta["n_genes_detected"] = (patient_counts > 0).sum(axis=1).to_numpy(dtype=int)

expected_all_design = {"GBM": 39, "LGG": 2, "ndGBM": 11, "rGBM": 5}
all_design_counts = design.groupby("condition")["source_backed_patient_id"].nunique().reindex(CONDITION_ORDER).to_dict()
if all_design_counts != expected_all_design:
    raise AssertionError(f"Source-backed design checkpoint failed: {all_design_counts}")

observed_counts = patient_meta.groupby("condition")["source_backed_patient_id"].nunique().reindex(CONDITION_ORDER, fill_value=0)
print("Notebook 22 qualifying patient pseudobulks by condition:")
display(observed_counts.rename("n_patients").reset_index())
print("Patient count matrix:", patient_counts.shape)


Notebook 22 qualifying patient pseudobulks by condition:


,condition,n_patients
0,GBM,39
1,LGG,2
2,ndGBM,10
3,rGBM,5


Patient count matrix: (56, 15081)


In [4]:
keep = gene_filter(patient_counts)
filtered_counts = patient_counts.loc[:, keep].copy()
logcpm = counts_to_logcpm(filtered_counts)
genes = filtered_counts.columns.tolist()

if not np.all(np.isfinite(logcpm.to_numpy())):
    raise AssertionError("Patient logCPM matrix contains non-finite values.")
if not np.allclose(patient_counts.to_numpy(), np.round(patient_counts.to_numpy())):
    raise AssertionError("Aggregated pseudobulk counts are not integer-like.")

global_rows = []
for gene in genes:
    groups = [logcpm.loc[patient_meta["condition"].eq(condition).to_numpy(), gene].to_numpy() for condition in CONDITION_ORDER]
    if any(len(group) == 0 for group in groups):
        continue
    test = stats.kruskal(*groups)
    row = {"gene": gene, "test": "Kruskal-Wallis", "n_conditions": 4, "H": float(test.statistic), "p_value": float(test.pvalue)}
    for condition, group in zip(CONDITION_ORDER, groups):
        row[f"n_{condition}"] = len(group)
        row[f"median_logCPM_{condition}"] = float(np.median(group))
    global_rows.append(row)
global_tests = pd.DataFrame(global_rows)
global_tests["q_value_bh"] = bh(global_tests["p_value"])
global_tests["interpretation"] = "exploratory_global_test_condition_dataset_confounding"
global_tests = global_tests.sort_values(["q_value_bh", "p_value", "gene"], na_position="last").reset_index(drop=True)

pairwise_rows = []
summary_rows = []
for condition_a, condition_b in combinations(CONDITION_ORDER, 2):
    mask_a = patient_meta["condition"].eq(condition_a).to_numpy()
    mask_b = patient_meta["condition"].eq(condition_b).to_numpy()
    n_a, n_b = int(mask_a.sum()), int(mask_b.sum())
    contrast = f"{condition_a}_vs_{condition_b}"
    same_dataset = set(patient_meta.loc[mask_a, "core_dataset"]) == set(patient_meta.loc[mask_b, "core_dataset"])
    confounding = not (condition_a in {"LGG", "ndGBM", "rGBM"} and condition_b in {"LGG", "ndGBM", "rGBM"})
    if n_a < MIN_PATIENTS_PER_TESTED_GROUP or n_b < MIN_PATIENTS_PER_TESTED_GROUP:
        summary_rows.append({"contrast": contrast, "condition_A": condition_a, "condition_B": condition_b,
                             "n_A": n_a, "n_B": n_b, "status": "not_tested_underpowered_group",
                             "condition_dataset_confounding": confounding, "n_genes_tested": 0})
        continue
    local = []
    for gene in genes:
        x = logcpm.loc[mask_a, gene].to_numpy(); y = logcpm.loc[mask_b, gene].to_numpy()
        local.append({"contrast": contrast, "condition_A": condition_a, "condition_B": condition_b,
                      "gene": gene, "n_A": n_a, "n_B": n_b,
                      "mean_logCPM_A": float(np.mean(x)), "mean_logCPM_B": float(np.mean(y)),
                      "median_logCPM_A": float(np.median(x)), "median_logCPM_B": float(np.median(y)),
                      "log2FC_A_vs_B": float(np.mean(x) - np.mean(y)),
                      "p_value_welch": safe_welch(x, y), "p_value_mannwhitney": safe_mannwhitney(x, y),
                      "rank_biserial_A_minus_B": rank_biserial(x, y),
                      "condition_dataset_confounding": confounding})
    local = pd.DataFrame(local)
    local["q_value_welch_bh"] = bh(local["p_value_welch"])
    local["q_value_mannwhitney_bh"] = bh(local["p_value_mannwhitney"])
    pairwise_rows.append(local)
    summary_rows.append({"contrast": contrast, "condition_A": condition_a, "condition_B": condition_b,
                         "n_A": n_a, "n_B": n_b, "status": "tested_exploratory",
                         "condition_dataset_confounding": confounding, "n_genes_tested": len(local),
                         "n_welch_fdr_0_05": int(local["q_value_welch_bh"].lt(0.05).sum()),
                         "n_mannwhitney_fdr_0_05": int(local["q_value_mannwhitney_bh"].lt(0.05).sum()),
                         "n_welch_fdr_0_05_abs_log2fc_0_5": int((local["q_value_welch_bh"].lt(0.05) & local["log2FC_A_vs_B"].abs().ge(0.5)).sum())})

pairwise = pd.concat(pairwise_rows, ignore_index=True) if pairwise_rows else pd.DataFrame()
pairwise_summary = pd.DataFrame(summary_rows)
display(pairwise_summary)


,contrast,condition_A,condition_B,n_A,n_B,status,condition_dataset_confounding,n_genes_tested,n_welch_fdr_0_05,n_mannwhitney_fdr_0_05,n_welch_fdr_0_05_abs_log2fc_0_5
0,GBM_vs_LGG,GBM,LGG,39,2,not_tested_underpowered_group,True,0,NaN,NaN,NaN
1,GBM_vs_ndGBM,GBM,ndGBM,39,10,tested_exploratory,True,15049,3152.0,2725.0,2643.0
2,GBM_vs_rGBM,GBM,rGBM,39,5,tested_exploratory,True,15049,1418.0,773.0,1216.0
3,LGG_vs_ndGBM,LGG,ndGBM,2,10,not_tested_underpowered_group,False,0,NaN,NaN,NaN
4,LGG_vs_rGBM,LGG,rGBM,2,5,not_tested_underpowered_group,False,0,NaN,NaN,NaN
5,ndGBM_vs_rGBM,ndGBM,rGBM,10,5,tested_exploratory,False,15049,0.0,0.0,0.0


In [5]:
top_rows = []
for contrast, group in pairwise.groupby("contrast", sort=True):
    ordered = group.sort_values(["q_value_welch_bh", "p_value_welch", "gene"], na_position="last")
    up = ordered.loc[ordered["log2FC_A_vs_B"] > 0].head(100).copy(); up["direction"] = "up_in_A"
    down = ordered.loc[ordered["log2FC_A_vs_B"] < 0].head(100).copy(); down["direction"] = "up_in_B"
    top_rows.extend([up, down])
top100 = pd.concat(top_rows, ignore_index=True) if top_rows else pd.DataFrame()

historical = pd.read_csv(ROOT / HISTORICAL_SUMMARY_REL, sep="	")
historical_contrast_col = "contrast"
comparison = pairwise_summary.merge(historical, on=historical_contrast_col, how="left", suffixes=("_patient", "_sample"))
comparison["patient_level_decision"] = np.where(
    comparison["status_patient"].eq("not_tested_underpowered_group"),
    "not_estimable_at_patient_level",
    np.where(comparison["condition_dataset_confounding"],
             "exploratory_due_to_condition_dataset_confounding",
             "patient_level_within_DS3_exploratory")
)
display(comparison)


,contrast,condition_A_patient,condition_B_patient,n_A_patient,n_B_patient,status_patient,condition_dataset_confounding,n_genes_tested_patient,n_welch_fdr_0_05,n_mannwhitney_fdr_0_05,n_welch_fdr_0_05_abs_log2fc_0_5,condition_A_sample,condition_B_sample,n_A_sample,n_B_sample,status_sample,n_genes_tested_sample,n_sig_welch_fdr_0_05,n_sig_mw_fdr_0_05,n_sig_abs_log2fc_0_5_fdr_0_05,patient_level_decision
0,GBM_vs_LGG,GBM,LGG,39,2,not_tested_underpowered_group,True,0,NaN,NaN,NaN,GBM,LGG,39,4,tested,15081,8504,6015,7455,not_estimable_at_patient_level
1,GBM_vs_ndGBM,GBM,ndGBM,39,10,tested_exploratory,True,15049,3152.0,2725.0,2643.0,GBM,ndGBM,39,18,tested,15081,4584,4192,3308,exploratory_due_to_condition_dataset_confounding
2,GBM_vs_rGBM,GBM,rGBM,39,5,tested_exploratory,True,15049,1418.0,773.0,1216.0,GBM,rGBM,39,16,tested,15081,5285,4556,4089,exploratory_due_to_condition_dataset_confounding
3,LGG_vs_ndGBM,LGG,ndGBM,2,10,not_tested_underpowered_group,False,0,NaN,NaN,NaN,LGG,ndGBM,4,18,tested,15081,6164,4569,5512,not_estimable_at_patient_level
4,LGG_vs_rGBM,LGG,rGBM,2,5,not_tested_underpowered_group,False,0,NaN,NaN,NaN,LGG,rGBM,4,16,tested,15081,6553,5270,5970,not_estimable_at_patient_level
5,ndGBM_vs_rGBM,ndGBM,rGBM,10,5,tested_exploratory,False,15049,0.0,0.0,0.0,ndGBM,rGBM,18,16,tested,15081,919,862,670,patient_level_within_DS3_exploratory


In [6]:
outputs = {}
write_tsv(patient_meta, "patient_pseudobulk_qc.tsv")
write_tsv(patient_counts, "patient_counts_pseudobulk.tsv.gz", index=True, compression={"method": "gzip", "mtime": 0})
write_tsv(logcpm, "patient_logCPM_pseudobulk.tsv.gz", index=True, compression={"method": "gzip", "mtime": 0})
write_tsv(global_tests, "patient_global_kruskal_all_genes.tsv")
write_tsv(pairwise_summary, "patient_pairwise_DE_test_summary.tsv")
write_tsv(pairwise, "patient_pairwise_DE_all_genes.tsv")
write_tsv(top100, "patient_pairwise_DE_top100_up_down.tsv")
write_tsv(comparison, "historical_sample_vs_patient_level_comparison.tsv")

readiness = pd.DataFrame([
    ("source_backed_design_patients", 57, design["source_backed_patient_id"].nunique()),
    ("patients_spanning_conditions", 0, int((design.groupby("source_backed_patient_id")["condition"].nunique() > 1).sum())),
    ("notebook22_sample_pseudobulks", 77, len(sample_counts)),
    ("sample_count_qc_sets_match", True, set(sample_counts.index.astype(str)) == set(sample_qc["sample_id"].astype(str))),
    ("patient_pseudobulk_rows", 56, len(patient_counts)),
    ("patient_ids_unique", True, not patient_meta["source_backed_patient_id"].duplicated().any()),
    ("pairwise_contrasts", 6, len(pairwise_summary)),
    ("tested_contrasts", 3, int(pairwise_summary["status"].eq("tested_exploratory").sum())),
    ("underpowered_LGG_contrasts", 3, int(pairwise_summary["status"].eq("not_tested_underpowered_group").sum())),
    ("canonical_files_modified", False, False),
], columns=["check", "expected", "observed"])
readiness["pass"] = readiness["expected"] == readiness["observed"]
write_tsv(readiness, "readiness_checks.tsv")

run_info = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().astimezone().isoformat(),
    "python": sys.version,
    "repository_root": str(ROOT),
    "output_directory": str(OUT_REL),
    "analysis_unit": "source-backed independent patient",
    "normalization": "log2(CPM + 1) after summing raw counts across all qualifying regional samples per patient",
    "gene_filter": f"CPM >= {MIN_CPM} in at least {MIN_PATIENTS_EXPRESSED} patient libraries",
    "minimum_patients_per_tested_group": MIN_PATIENTS_PER_TESTED_GROUP,
    "canonical_files_modified": False,
}
run_path = OUT_DIR / "run_info.json"
run_path.write_text(json.dumps(run_info, indent=2), encoding="utf-8")
outputs[run_path.name] = run_path

manifest_rows = []
for name, path in sorted(outputs.items()):
    rows = columns = np.nan
    if path.name.endswith(".tsv"):
        rows, columns = pd.read_csv(path, sep="	").shape
    elif path.name.endswith(".tsv.gz"):
        rows, columns = pd.read_csv(path, sep="	", compression="gzip").shape
    data = path.read_bytes()
    manifest_rows.append({"file": name, "relative_path": str(path.relative_to(ROOT)), "rows": rows,
                          "columns": columns, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()})
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUT_DIR / "output_manifest.tsv", sep="	", index=False)

display(readiness)
display(manifest)
if not readiness["pass"].all():
    raise AssertionError("Final patient-level pseudobulk readiness checks failed.")
print("PATIENT-LEVEL MALIGNANT PSEUDOBULK CONDITION REANALYSIS PASSED")
print("Outputs:", OUT_DIR)


,check,expected,observed,pass
0,source_backed_design_patients,57,57,True
1,patients_spanning_conditions,0,0,True
2,notebook22_sample_pseudobulks,77,77,True
3,sample_count_qc_sets_match,True,True,True
4,patient_pseudobulk_rows,56,56,True
5,patient_ids_unique,True,True,True
6,pairwise_contrasts,6,6,True
7,tested_contrasts,3,3,True
8,underpowered_LGG_contrasts,3,3,True
9,canonical_files_modified,False,False,True


,file,relative_path,rows,columns,bytes,sha256
0,historical_sample_vs_patient_level_comparison.tsv,metadata\sensitivity\patient_level_malignant_p...,6.0,21.0,1319,a23885df7576fa7323ea707df61806ba035c1026cb4f58...
1,patient_counts_pseudobulk.tsv.gz,metadata\sensitivity\patient_level_malignant_p...,56.0,15082.0,1356450,06265c2e407719df0a13288eae558086d59129877b6388...
2,patient_global_kruskal_all_genes.tsv,metadata\sensitivity\patient_level_malignant_p...,15049.0,15.0,3304458,61988e7cc846d3050c7cf65f6e92fed2f0bb14b6051cde...
3,patient_logCPM_pseudobulk.tsv.gz,metadata\sensitivity\patient_level_malignant_p...,56.0,15050.0,3555315,74d68b5fa3ed1ef027dbb72e59ab5b9007d99aa6de3889...
4,patient_pairwise_DE_all_genes.tsv,metadata\sensitivity\patient_level_malignant_p...,45147.0,17.0,10175109,dfb77d23e9306e6ca532892ebc332f20da0ab8ec6286b2...
5,patient_pairwise_DE_test_summary.tsv,metadata\sensitivity\patient_level_malignant_p...,6.0,11.0,600,d64681b6b88a3ce1ad68f7e64be1e04947242b107bfafd...
6,patient_pairwise_DE_top100_up_down.tsv,metadata\sensitivity\patient_level_malignant_p...,600.0,18.0,145140,ea6bea38f21338fe9d0c55e75811f9ac2d1525bdc23ce0...
7,patient_pseudobulk_qc.tsv,metadata\sensitivity\patient_level_malignant_p...,56.0,8.0,4642,80386ff4952e4b54ff1e3a76b41116773089de2cd6919f...
8,readiness_checks.tsv,metadata\sensitivity\patient_level_malignant_p...,10.0,4.0,403,54b626e92b16b8553b64c38b7df589b20c022b1b9c1701...
9,run_info.json,metadata\sensitivity\patient_level_malignant_p...,NaN,NaN,737,d72d11224b8f7025e4ca761a6a32d37b817fd2a8362cb2...


PATIENT-LEVEL MALIGNANT PSEUDOBULK CONDITION REANALYSIS PASSED
Outputs: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\patient_level_malignant_pseudobulk_condition_reanalysis


## Interpretation rules

- The patient is the independent replicate; regional samples from the same patient are never treated as independent observations.
- Contrasts containing LGG are not tested because only two independent LGG patients are available.
- GBM-versus-DS3-condition contrasts are explicitly marked as condition–dataset confounded and cannot establish a condition-specific causal effect.
- The ndGBM-versus-rGBM comparison is within DS3 and is the most interpretable tested contrast, but remains exploratory because rGBM contains only five patients.
- Canonical Notebook 22 outputs are not modified by this notebook.
